# Andrew Fox - Hydrodynamic Modeling of the VideoRay Defender 

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import sklearn
import sys
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import torch.optim as optim
import torch.nn.functional as F
from bayesian_torch.models.dnn_to_bnn import get_kl_loss
from bayesian_torch.layers.variational_layers import LinearReparameterization
import hamiltorch
from scipy.signal import butter, filtfilt

print("Library Versions:")
print('numpy:',np.__version__)
print('pandas:',pd.__version__)
print('torch:',torch.__version__)
print('sklearn:',sklearn.__version__)
print(torch.cuda.is_available())          # Should print: True
print(torch.cuda.get_device_name(0))      # Should print: your GPU model


Library Versions:
numpy: 1.23.5
pandas: 2.3.2
torch: 2.8.0+cu128
sklearn: 1.7.2
True
NVIDIA GeForce RTX 5070 Ti Laptop GPU


In [3]:
n_epochs = 10000 # Number of epochs for training or number of iterations for optimization
verbose_option = True # Controls whether to print training progress

# Input/Output Data for the VideoRay Defender

In [171]:

# Read the dataset
rov = pd.read_csv(
    "/home/andrew/fossen_ml_pipeline/defender_parameter_estimator/csv_files/N_Data/Tank_data/defender_data_n_run_1_only_mocap_data_savgol.csv",
    sep="\t"
)

# Drop the first 3 rows
rov = rov.iloc[3:].reset_index(drop=True)

# -------------------------------
# Columns to require (no NaNs)
# -------------------------------
signal_cols = [
    "u_dot","v_dot","w_dot","p_dot","q_dot","r_dot",
    "u","v","w","p","q","r",
    "x","y","z","phi","theta","psi",
    "X","Y","Z","K","M","N",
]
signal_cols = [c for c in signal_cols if c in rov.columns]

# If you want to ALSO require the valid flags (optional)
flag_cols = [c for c in ["pose_valid", "twist_valid", "wrench_valid"] if c in rov.columns]
age_cols  = [c for c in ["pose_age", "twist_age", "wrench_age"] if c in rov.columns]

# Make flags ints (in case they come in as floats)
for c in flag_cols:
    rov[c] = rov[c].fillna(0).astype(int)

# -------------------------------
# 1) DROP ROWS WITH NaNs
# -------------------------------
before = len(rov)
rov = rov.dropna(subset=signal_cols).reset_index(drop=True)
after = len(rov)
print(f"Dropped {before - after} rows due to NaNs in required signal columns.")

# (Optional) if you want to be extra strict and only keep rows where mocap pose/twist are valid:
# if "twist_valid" in rov.columns:
#     rov = rov[rov["twist_valid"] == 1].reset_index(drop=True)
# if "pose_valid" in rov.columns:
#     rov = rov[rov["pose_valid"] == 1].reset_index(drop=True)

# -------------------------------
# 2) Epsilon-zero + rounding
# -------------------------------
epsilon = 1e-4

arr = rov[signal_cols].to_numpy(dtype=float)

# No NaNs should remain, but keep this safe anyway
finite = np.isfinite(arr)
arr[finite & (np.abs(arr) < epsilon)] = 0.0
arr[finite] = np.round(arr[finite], 5)

rov[signal_cols] = arr

# Keep ages tidy (optional)
for c in age_cols:
    rov[c] = rov[c].astype(float).round(6)

# Print cleaned result
print("Cleaned shape:", rov.shape)
print("First few rows:\n", rov.head())

# Sanity check: should be 0 now
print("\nNaNs per column (top 10):")
print(rov.isna().sum().sort_values(ascending=False).head(10))


Dropped 0 rows due to NaNs in required signal columns.
Cleaned shape: (40117, 25)
First few rows:
            time    u_dot    v_dot    w_dot    p_dot    q_dot    r_dot  \
0  1.764789e+09  0.06807 -0.30849 -0.35826  0.75279  0.21283  0.88716   
1  1.764789e+09  0.06730 -0.30373 -0.35779  0.78390  0.20460  0.85596   
2  1.764789e+09  0.06842 -0.30732 -0.36728  0.83836  0.20199  0.84780   
3  1.764789e+09  0.09513 -0.42502 -0.51543  1.22486  0.27236  1.14640   
4  1.764789e+09  0.06130 -0.27223 -0.33507  0.82844  0.16997  0.71700   

         u        v        w  ...        z      phi    theta      psi    X  \
0 -0.13050  0.08562  0.13811  ... -1.24941  0.01744 -0.09942 -2.44378  0.0   
1 -0.12945  0.08088  0.13257  ... -1.24808  0.01431 -0.09976 -2.44622  0.0   
2 -0.12844  0.07634  0.12718  ... -1.24562  0.00770 -0.09948 -2.45075  0.0   
3 -0.12747  0.07200  0.12195  ... -1.24438  0.00535 -0.09856 -2.45278  0.0   
4 -0.12654  0.06784  0.11687  ... -1.24329  0.00170 -0.09882 -2.45501  0

In [4]:
# ==== RANK CHECK =====

# Extract surge velocity and acceleration
u_dot = rov["u_dot"].to_numpy()
u     = rov["u"].to_numpy()

# Build surge regressor matrix Phi_X
Phi_X = np.column_stack([
    u_dot,
    u,
    np.abs(u) * u
])  # shape (N, 3)

print("Phi_X shape:", Phi_X.shape)

rank = np.linalg.matrix_rank(Phi_X)
print("Rank of Phi_X:", rank)

U, S, Vt = np.linalg.svd(Phi_X, full_matrices=False)
print("Singular values:", S)
print("Condition number:", S[0] / S[-1])
print("Smallest-singular direction Vt[-1]:", Vt[-1])



Phi_X shape: (60118, 3)
Rank of Phi_X: 3
Singular values: [744.66374703  12.09632452   1.22719778]
Condition number: 606.800110665422
Smallest-singular direction Vt[-1]: [-1.61541869e-04 -1.61508916e-01  9.86871240e-01]


In [47]:

# ==== DVL ISSUE DATA CLEANUP ===== #


# ====== Remove DVL dropouts (NaNs in u, v, w) ======
required_dvl_cols = ["u", "v", "w"]
rov = rov.dropna(subset=required_dvl_cols).reset_index(drop=True)

print("After removing DVL-dropout rows:", rov.shape)

# Drop time column
rov_numerics = rov.drop(columns=["time"])

# Convert to NumPy
rov_np = rov_numerics.to_numpy(dtype=float)

# Train/test split
rov_train_np, rov_test_np = train_test_split(
    rov_np, test_size=0.25, random_state=42
)

After removing DVL-dropout rows: (38668, 25)


In [88]:

#### ====== MASTER CSV CLEANER SCRIPT ======== ####


###############################################################
#        DEFENDER CSV → CLEANED NUMPY TRAIN/TEST SETS
#
# This block calls `shape_defender_data()` to:
#   1. Remove rows where the DVL (u, v, w) signal has dropped out.
#   2. Remove rows containing ANY NaNs in the physics state
#      (ν, ν̇, η). Ensures only valid sensor windows remain.
#   3. Zero-out all DOFs *except* the ones explicitly chosen
#      via `active_dofs=[...]`.
#         - Use ["Z"] for heave-only modeling.
#         - Use ["X"] for surge-only modeling.
#         - Use ["X","Y","Z"] for planar 3DOF.
#         - Use "ALL" to keep the full 6DOF dataset.
#   4. Drop the time column and convert the cleaned dataframe
#      into NumPy arrays.
#   5. Perform a 75/25 train/test split (configurable).
#
# Returned:
#     rov_train_np  -- cleaned training data (NumPy)
#     rov_test_np   -- cleaned testing data  (NumPy)
###############################################################

def shape_defender_data(
    rov,
    drop_dvl=True,
    drop_all_nans=True,
    active_dofs="ALL",     # "ALL" or list like ["Z"], ["X","Z"], etc.
    test_size=0.25,
    random_state=42
):
    """
    Shapes Defender CSV data for ML training:
    - Drops rows where DVL is missing
    - Drops rows with ANY NaNs in physics columns
    - Zeroes out inactive DOFs
    - Returns numpy train/test arrays
    """

    # ----------------------------------------------------------
    # 1. Drop DVL-missing rows
    # ----------------------------------------------------------
    if drop_dvl:
        required_dvl_cols = ["u", "v", "w"]
        rov = rov.dropna(subset=required_dvl_cols)
        print(f"[DVL Filter] After removing DVL NaNs: {rov.shape}")

    # ----------------------------------------------------------
    # 2. Drop rows where ANY ν̇, ν, η contains NaNs
    # ----------------------------------------------------------
    if drop_all_nans:
        physics_cols = [col for col in rov.columns if col not in ["time"]]
        before = len(rov)
        rov = rov.dropna(subset=physics_cols)
        after = len(rov)
        print(f"[NaN Filter] Removed {before - after} rows with NaNs in physics state.")
        print(f"[NaN Filter] Dataset after state-clean: {rov.shape}")

    # ----------------------------------------------------------
    # 3. Zero out non-active DOFs
    # ----------------------------------------------------------
    # DOF column mapping
    dof_map = {
        "X": ["u", "u_dot"],
        "Y": ["v", "v_dot"],
        "Z": ["w", "w_dot"],
        "K": ["p", "p_dot"],
        "M": ["q", "q_dot"],
        "N": ["r", "r_dot"],
    }

    all_dofs = list(dof_map.keys())

    # Normalize input
    if active_dofs == "ALL":
        active_dofs = all_dofs
    elif isinstance(active_dofs, str):
        active_dofs = [active_dofs]

    # Validate
    for d in active_dofs:
        if d not in all_dofs:
            raise ValueError(f"Invalid DOF '{d}'. Must be one of {all_dofs}.")

    inactive_dofs = [d for d in all_dofs if d not in active_dofs]

    # Perform zeroing
    zeroed_columns = []
    for d in inactive_dofs:
        for col in dof_map[d]:
            if col in rov.columns:
                rov[col] = 0.0
                zeroed_columns.append(col)

    # Enhanced printout
    print(f"[DOF Filter] Requested active DOFs: {active_dofs}")
    print(f"[DOF Filter] Inactive DOFs zeroed: {inactive_dofs}")
    print(f"[DOF Filter] Columns zeroed: {zeroed_columns if zeroed_columns else 'None'}")

    # ----------------------------------------------------------
    # 4. Drop time & convert to NumPy
    # ----------------------------------------------------------
    if "time" in rov.columns:
        rov_numeric = rov.drop(columns=["time"])
    else:
        rov_numeric = rov.copy()

    rov_np = rov_numeric.to_numpy(dtype=float)

    # ----------------------------------------------------------
    # 5. Train/test split
    # ----------------------------------------------------------
    rov_train_np, rov_test_np = train_test_split(
        rov_np, test_size=test_size, random_state=random_state
    )

    print(f"[Split] Train shape: {rov_train_np.shape}, Test shape: {rov_test_np.shape}")

    return rov_train_np, rov_test_np

rov_train_np, rov_test_np = shape_defender_data(
    rov,
    drop_dvl=True,
    drop_all_nans=True,
    active_dofs="Z" #ALL
)

[DVL Filter] After removing DVL NaNs: (67564, 31)
[NaN Filter] Removed 0 rows with NaNs in physics state.
[NaN Filter] Dataset after state-clean: (67564, 31)
[DOF Filter] Requested active DOFs: ['Z']
[DOF Filter] Inactive DOFs zeroed: ['X', 'Y', 'K', 'M', 'N']
[DOF Filter] Columns zeroed: ['u', 'u_dot', 'v', 'v_dot', 'p', 'p_dot', 'q', 'q_dot', 'r', 'r_dot']
[Split] Train shape: (50673, 30), Test shape: (16891, 30)


In [172]:
# === Clean and split ROV dataset === REAL WORLD DEFENDER DATA PARSING

# Drop non-numeric / metadata columns
rov_numerics = rov.drop(columns=["time"])


# Convert to NumPy (force float dtype)
rov_np = rov_numerics.to_numpy(dtype=float)

# Train/test split
rov_train_np, rov_test_np = train_test_split(rov_np, test_size=0.25, random_state=42)


In [64]:
# === Clean and split ROV dataset === SIMULATOR DATA PARSING

# Drop non-numeric / metadata columns
rov_numerics = rov.drop(columns=["time", "norm_dof", "norm_value"], errors="ignore")


# Convert to NumPy (force float dtype)
rov_np = rov_numerics.to_numpy(dtype=float)

# Train/test split
rov_train_np, rov_test_np = train_test_split(rov_np, test_size=0.25, random_state=42)


print("rov_train_np shape:", rov_train_np.shape)
print("rov_numerics columns:", len(rov_numerics.columns))
print("rov_numerics last 10 cols:", rov_numerics.columns[-10:].tolist())


rov_train_np shape: (51948, 24)
rov_numerics columns: 24
rov_numerics last 10 cols: ['z', 'phi', 'theta', 'psi', 'X', 'Y', 'Z', 'K', 'M', 'N']


In [173]:
# === Full input data ===
x_train_np = rov_train_np[:, :18]
y_train_np = rov_train_np[:, 18:24]

x_test_np = rov_test_np[:, :18]
y_test_np = rov_test_np[:, 18:24]

# === Split x into [nu_dot, nu, eta] ===
nu_dot_train = x_train_np[:, 0:6]
nu_train     = x_train_np[:, 6:12]
eta_train    = x_train_np[:, 12:18]

nu_dot_test = x_test_np[:, 0:6]
nu_test     = x_test_np[:, 6:12]
eta_test    = x_test_np[:, 12:18]


In [174]:
# Convert to PyTorch tensors
nu_dot_train = torch.tensor(nu_dot_train, dtype=torch.float32)
nu_train     = torch.tensor(nu_train,     dtype=torch.float32)
eta_train    = torch.tensor(eta_train,    dtype=torch.float32)
y_train      = torch.tensor(y_train_np,   dtype=torch.float32)

nu_dot_test = torch.tensor(nu_dot_test, dtype=torch.float32)
nu_test     = torch.tensor(nu_test,     dtype=torch.float32)
eta_test    = torch.tensor(eta_test,    dtype=torch.float32)
y_test      = torch.tensor(y_test_np,   dtype=torch.float32)


# MLE ROV DYNAMICS NODE - BOLEAN FOR LEARNABLE MOMENT OF INERTIA

In [120]:

"""
    Fossen-style inverse dynamics model:

        tau = M(nu_dot) + C(nu)nu + D(nu)nu + g(eta)

    with toggles to enable/disable individual components so MLE/MAP/HMC
    all share the exact same implementation.

    Notes:
    - NED convention (z down) as in your current code.
    - Added-mass coefficients are used in M_A always (as you currently do).
    - C_A (added-mass Coriolis) can be toggled off to avoid Munk-moment-induced instability
      when off-diagonal damping is not modeled.
    """

class ROVDynamicsModel(nn.Module):
    def __init__(
        self,
        use_Crb: bool = True,
        use_Ca: bool = True,
        use_D: bool = True,
        use_g: bool = True,
        m_val: float = 23.89,

        # NEW:
        learn_inertia: bool = False,
        Ixx_cad: float = 0.393,
        Iyy_cad: float = 1.302,
        Izz_cad: float = 1.429,
    ):
        super().__init__()

        self.use_Crb = use_Crb
        self.use_Ca  = use_Ca
        self.use_D   = use_D
        self.use_g   = use_g

        # --------------------------
        # (1) Added-Mass Coefficients
        # --------------------------
        self.X_dot_u = nn.Parameter(torch.tensor(-0.0))
        self.Y_dot_v = nn.Parameter(torch.tensor(-0.0))
        self.Z_dot_w = nn.Parameter(torch.tensor(-0.0))
        self.K_dot_p = nn.Parameter(torch.tensor(-0.0))
        self.M_dot_q = nn.Parameter(torch.tensor(-0.0))
        self.N_dot_r = nn.Parameter(torch.tensor(-0.0))

        # --------------------------
        # (2) Rigid-Body Inertias
        # --------------------------
        self.learn_inertia = bool(learn_inertia)

        # Store CAD inertias as buffers so they move with .to(device)
        # and appear in state_dict, but are NOT trainable params.
        self.register_buffer("Ixx_cad", torch.tensor(float(Ixx_cad), dtype=torch.float32))
        self.register_buffer("Iyy_cad", torch.tensor(float(Iyy_cad), dtype=torch.float32))
        self.register_buffer("Izz_cad", torch.tensor(float(Izz_cad), dtype=torch.float32))

        if self.learn_inertia:
            self.I_xx = nn.Parameter(self.Ixx_cad.clone())
            self.I_yy = nn.Parameter(self.Iyy_cad.clone())
            self.I_zz = nn.Parameter(self.Izz_cad.clone())
        else:
            # Not Parameters → optimizer can’t change them
            self.I_xx = None
            self.I_yy = None
            self.I_zz = None

        # --------------------------
        # (3) Center of Gravity Offsets
        # --------------------------
        self.x_g = nn.Parameter(torch.tensor(0.0))
        self.y_g = nn.Parameter(torch.tensor(0.0))
        self.z_g = nn.Parameter(torch.tensor(0.0))

        # --------------------------
        # (4) Linear Damping
        # --------------------------
        self.X_u = nn.Parameter(torch.tensor(0.0))
        self.Y_v = nn.Parameter(torch.tensor(0.0))
        self.Z_w = nn.Parameter(torch.tensor(0.0))
        self.K_p = nn.Parameter(torch.tensor(0.0))
        self.M_q = nn.Parameter(torch.tensor(0.0))
        self.N_r = nn.Parameter(torch.tensor(0.0))

        # --------------------------
        # (5) Quadratic Damping
        # --------------------------
        self.X_uu = nn.Parameter(torch.tensor(0.0))
        self.Y_vv = nn.Parameter(torch.tensor(0.0))
        self.Z_ww = nn.Parameter(torch.tensor(0.0))
        self.K_pp = nn.Parameter(torch.tensor(0.0))
        self.M_qq = nn.Parameter(torch.tensor(0.0))
        self.N_rr = nn.Parameter(torch.tensor(0.0))

        # --------------------------
        # (6) Restoring / Buoyancy
        # --------------------------
        self.B   = nn.Parameter(torch.tensor(0.0))
        self.z_b = nn.Parameter(torch.tensor(-0.0))

        # --------------------------
        # (7) Constant Mass
        # --------------------------
        self.m_val = float(m_val)

        self.Crb = None
        self.Ca  = None
        self.C   = None
        self.D   = None

    # --- NEW helper ---
    def _get_inertias(self, device, dtype=torch.float32):
        """
        Returns (Ixx, Iyy, Izz) as tensors on the requested device/dtype.
        Uses learned params if enabled, otherwise CAD buffers.
        """
        if self.learn_inertia:
            Ixx = self.I_xx.to(device=device, dtype=dtype)
            Iyy = self.I_yy.to(device=device, dtype=dtype)
            Izz = self.I_zz.to(device=device, dtype=dtype)
        else:
            Ixx = self.Ixx_cad.to(device=device, dtype=dtype)
            Iyy = self.Iyy_cad.to(device=device, dtype=dtype)
            Izz = self.Izz_cad.to(device=device, dtype=dtype)
        return Ixx, Iyy, Izz

    # -------------------------------------------------------------------------
    # MASS MATRIX
    # -------------------------------------------------------------------------
    def build_mass_matrix(self) -> torch.Tensor:
        """
        M = M_RB + M_A (6x6)
        """
        # Added mass (note your convention: Ma = -diag([X_dot_u, ...]))
        Ma = -torch.diag(torch.stack([
            self.X_dot_u, self.Y_dot_v, self.Z_dot_w,
            self.K_dot_p, self.M_dot_q, self.N_dot_r
        ]))

        m  = torch.as_tensor(self.m_val, dtype=torch.float32, device=Ma.device)
        xg = self.x_g
        yg = self.y_g
        zg = self.z_g

        # NEW: get inertias (learned or CAD)
        Ixx, Iyy, Izz = self._get_inertias(device=Ma.device, dtype=torch.float32)

        Mrb = torch.zeros((6, 6), dtype=torch.float32, device=Ma.device)

        Mrb[0, 0] = m
        Mrb[1, 1] = m
        Mrb[2, 2] = m

        Mrb[0, 4] =  m * zg
        Mrb[0, 5] = -m * yg
        Mrb[1, 3] = -m * zg
        Mrb[1, 5] =  m * xg
        Mrb[2, 3] =  m * yg
        Mrb[2, 4] = -m * xg

        Mrb[3, 1] = -m * zg
        Mrb[3, 2] =  m * yg
        Mrb[3, 3] = Ixx

        Mrb[4, 0] =  m * zg
        Mrb[4, 2] = -m * xg
        Mrb[4, 4] = Iyy

        Mrb[5, 0] = -m * yg
        Mrb[5, 1] =  m * xg
        Mrb[5, 5] = Izz

        return Mrb + Ma

    # -------------------------------------------------------------------------
    # CORIOLIS: C = C_RB + C_A (with toggles)
    # -------------------------------------------------------------------------
    def build_coriolis(self, nu: torch.Tensor) -> torch.Tensor:
        """
        Returns C_total with flags controlling inclusion of Crb and Ca.
        nu: (B,6)
        """
        u, v, w, p, q, r = [nu[:, i] for i in range(6)]
        Bsz = nu.shape[0]

        m = torch.as_tensor(self.m_val, dtype=torch.float32, device=nu.device)

        Ixx, Iyy, Izz = self._get_inertias(device=nu.device, dtype=torch.float32)

        xg = self.x_g
        yg = self.y_g
        zg = self.z_g

        # --- Build Crb ---
        Crb = torch.zeros((Bsz, 6, 6), dtype=torch.float32, device=nu.device)

        Crb[:, 0, 1] = -m * r
        Crb[:, 0, 2] =  m * q
        Crb[:, 0, 3] =  m * (q * yg + r * zg)
        Crb[:, 0, 4] = -m * (q * xg)
        Crb[:, 0, 5] = -m * (r * xg)

        Crb[:, 1, 0] =  m * r
        Crb[:, 1, 2] = -m * p
        Crb[:, 1, 3] = -m * (p * yg)
        Crb[:, 1, 4] =  m * (p * xg + r * zg)
        Crb[:, 1, 5] = -m * (r * yg)

        Crb[:, 2, 0] = -m * q
        Crb[:, 2, 1] =  m * p
        Crb[:, 2, 3] = -m * (p * zg)
        Crb[:, 2, 4] = -m * (q * zg)
        Crb[:, 2, 5] =  m * (p * xg + q * yg)

        Crb[:, 3, 0] = -m * (q * yg + r * zg)
        Crb[:, 3, 1] =  m * (p * yg)
        Crb[:, 3, 2] =  m * (p * zg)
        Crb[:, 3, 4] =  Izz * r
        Crb[:, 3, 5] = -Iyy * q

        Crb[:, 4, 0] =  m * (q * xg)
        Crb[:, 4, 1] = -m * (p * xg + r * zg)
        Crb[:, 4, 2] =  m * (q * zg)
        Crb[:, 4, 3] = -Izz * r
        Crb[:, 4, 5] =  Ixx * p

        Crb[:, 5, 0] =  m * (r * xg)
        Crb[:, 5, 1] =  m * (r * yg)
        Crb[:, 5, 2] = -m * (p * xg + q * yg)
        Crb[:, 5, 3] =  Iyy * q
        Crb[:, 5, 4] = -Ixx * p

        # --- Build Ca (optional) ---
        Ca = torch.zeros_like(Crb)
        if self.use_Ca:
            a1 = self.X_dot_u * u
            a2 = self.Y_dot_v * v
            a3 = self.Z_dot_w * w
            b1 = self.K_dot_p * p
            b2 = self.M_dot_q * q
            b3 = self.N_dot_r * r

            Ca[:, 0, 4] = -a3
            Ca[:, 0, 5] =  a2
            Ca[:, 1, 3] =  a3
            Ca[:, 1, 5] = -a1
            Ca[:, 2, 3] = -a2
            Ca[:, 2, 4] =  a1

            Ca[:, 3, 1] = -a3
            Ca[:, 3, 2] =  a2
            Ca[:, 3, 4] = -b3
            Ca[:, 3, 5] =  b2

            Ca[:, 4, 0] =  a3
            Ca[:, 4, 2] = -a1
            Ca[:, 4, 3] =  b3
            Ca[:, 4, 5] = -b1

            Ca[:, 5, 0] = -a2
            Ca[:, 5, 1] =  a1
            Ca[:, 5, 3] = -b2
            Ca[:, 5, 4] =  b1

        # --- Combine per flags ---
        C_total = torch.zeros_like(Crb)
        if self.use_Crb:
            C_total = C_total + Crb
        if self.use_Ca:
            C_total = C_total + Ca

        # Store for debug
        self.Crb = Crb
        self.Ca  = Ca
        self.C   = C_total

        return C_total

    # -------------------------------------------------------------------------
    # DAMPING
    # -------------------------------------------------------------------------
    def build_damping(self, nu: torch.Tensor) -> torch.Tensor:
        """
        D(ν) = -diag(lin + quad*|nu|)  (batch,6,6)
        """
        lin_params = torch.stack([self.X_u, self.Y_v, self.Z_w, self.K_p, self.M_q, self.N_r])
        quad_params = torch.stack([self.X_uu, self.Y_vv, self.Z_ww, self.K_pp, self.M_qq, self.N_rr])

        diag_entries = lin_params.unsqueeze(0) + quad_params.unsqueeze(0) * torch.abs(nu)
        D = -1.0 * torch.diag_embed(diag_entries)

        self.D = D
        return D

    # -------------------------------------------------------------------------
    # RESTORING FORCE
    # -------------------------------------------------------------------------
    def build_restoring_force(self, eta: torch.Tensor) -> torch.Tensor:
        """
        g(eta) in NED (z down)
        eta: (B,6) [x,y,z,phi,theta,psi]
        """
        phi   = eta[:, 3]
        theta = eta[:, 4]

        g0 = 9.8
        m = torch.as_tensor(self.m_val, dtype=torch.float32, device=eta.device)
        W = m * g0
        B = self.B.to(dtype=torch.float32, device=eta.device)

        x_G, y_G, z_G = self.x_g, self.y_g, self.z_g
        x_B = torch.zeros(1, device=eta.device)
        y_B = torch.zeros(1, device=eta.device)
        z_B = self.z_b.to(dtype=torch.float32, device=eta.device)

        WB    = W - B
        xW_xB = x_G * W - x_B * B
        yW_yB = y_G * W - y_B * B
        zW_zB = z_G * W - z_B * B

        tau_g = torch.stack([
            WB * torch.sin(theta),
            -WB * torch.cos(theta) * torch.sin(phi),
            -WB * torch.cos(theta) * torch.cos(phi),
            -yW_yB * torch.cos(theta) * torch.cos(phi) + zW_zB * torch.cos(theta) * torch.sin(phi),
            zW_zB * torch.sin(theta) + xW_xB * torch.cos(theta) * torch.cos(phi),
            -xW_xB * torch.cos(theta) * torch.sin(phi) - yW_yB * torch.sin(theta)
        ], dim=1)

        return tau_g

    # -------------------------------------------------------------------------
    # FORWARD (inverse dynamics prediction)
    # -------------------------------------------------------------------------
    def forward(self, nu_dot: torch.Tensor, nu: torch.Tensor, eta: torch.Tensor) -> torch.Tensor:
        """
        nu_dot: (B,6)
        nu:     (B,6)
        eta:    (B,6)
        returns tau: (B,6)
        """
        # Mass
        M = self.build_mass_matrix()                 # (6,6)
        tau_M = torch.matmul(nu_dot, M.T)            # (B,6)

        # Coriolis (optional pieces inside)
        C = self.build_coriolis(nu)                  # (B,6,6)
        tau_C = torch.einsum("bij,bj->bi", C, nu)    # (B,6)

        # Damping
        if self.use_D:
            D = self.build_damping(nu)               # (B,6,6)
            tau_D = torch.einsum("bij,bj->bi", D, nu)
        else:
            tau_D = torch.zeros_like(tau_M)

        # Restoring
        if self.use_g:
            tau_g = self.build_restoring_force(eta)  # (B,6)
        else:
            tau_g = torch.zeros_like(tau_M)

        return tau_M + tau_C + tau_D + tau_g

    # -------------------------------------------------------------------------
    # REGULARIZATION HELPERS (unchanged)
    # -------------------------------------------------------------------------
    def L2reg(self, selected_names, prior_means):
        loss = 0.0
        for name, param in self.named_parameters():
            if name in selected_names:
                mu = prior_means[name].to(param.device)
                loss += torch.sum((param - mu) ** 2)
        return loss

    def L1reg(self, param_names):
        l1reg_sum = 0.0
        for name, param in self.named_parameters():
            if name in param_names:
                l1reg_sum += torch.sum(torch.abs(param))
        return l1reg_sum


# MAP ROV DYNAMICS NODE - ENFORCES LINEAR DAMPING TO BE NEGATIVE

In [175]:

"""
    Fossen-style inverse dynamics model:

        tau = M(nu_dot) + C(nu)nu + D(nu)nu + g(eta)

    with toggles to enable/disable individual components so MLE/MAP/HMC
    all share the exact same implementation.

    Notes:
    - NED convention (z down) as in your current code.
    - Added-mass coefficients are used in M_A always (as you currently do).
    - C_A (added-mass Coriolis) can be toggled off to avoid Munk-moment-induced instability
      when off-diagonal damping is not modeled.
    """

class ROVDynamicsModel(nn.Module):
    def __init__(
        self,
        use_Crb: bool = True,
        use_Ca: bool = True,
        use_D: bool = True,
        use_g: bool = True,
        m_val: float = 23.89,

        # NEW:
        learn_inertia: bool = True,
        Ixx_cad: float = 0.393,
        Iyy_cad: float = 1.302,
        Izz_cad: float = 1.429,
    ):
        super().__init__()

        self.use_Crb = use_Crb
        self.use_Ca  = use_Ca
        self.use_D   = use_D
        self.use_g   = use_g

        # --------------------------
        # (1) Added-Mass Coefficients
        # --------------------------
        self.X_dot_u = nn.Parameter(torch.tensor(-0.0))
        self.Y_dot_v = nn.Parameter(torch.tensor(-0.0))
        self.Z_dot_w = nn.Parameter(torch.tensor(-0.0))
        self.K_dot_p = nn.Parameter(torch.tensor(-0.0))
        self.M_dot_q = nn.Parameter(torch.tensor(-0.0))
        self.N_dot_r = nn.Parameter(torch.tensor(-0.0))

        # --------------------------
        # (2) Rigid-Body Inertias
        # --------------------------
        self.learn_inertia = bool(learn_inertia)

        # Store CAD inertias as buffers so they move with .to(device)
        # and appear in state_dict, but are NOT trainable params.
        self.register_buffer("Ixx_cad", torch.tensor(float(Ixx_cad), dtype=torch.float32))
        self.register_buffer("Iyy_cad", torch.tensor(float(Iyy_cad), dtype=torch.float32))
        self.register_buffer("Izz_cad", torch.tensor(float(Izz_cad), dtype=torch.float32))

        if self.learn_inertia:
            self.I_xx = nn.Parameter(self.Ixx_cad.clone())
            self.I_yy = nn.Parameter(self.Iyy_cad.clone())
            self.I_zz = nn.Parameter(self.Izz_cad.clone())
        else:
            # Not Parameters → optimizer can’t change them
            self.I_xx = None
            self.I_yy = None
            self.I_zz = None

        # --------------------------
        # (3) Center of Gravity Offsets
        # --------------------------
        self.x_g = nn.Parameter(torch.tensor(0.0))
        self.y_g = nn.Parameter(torch.tensor(0.0))
        self.z_g = nn.Parameter(torch.tensor(0.0))

        # --------------------------
        # (4) Linear Damping
        # --------------------------
        self.X_u = nn.Parameter(torch.tensor(-3.0))
        self.Y_v = nn.Parameter(torch.tensor(-3.0))
        self.Z_w = nn.Parameter(torch.tensor(-3.0))
        self.K_p = nn.Parameter(torch.tensor(-1.0))
        self.M_q = nn.Parameter(torch.tensor(-1.0))
        self.N_r = nn.Parameter(torch.tensor(-1.0))

        # --------------------------
        # (5) Quadratic Damping
        # --------------------------
        self.X_uu = nn.Parameter(torch.tensor(0.0))
        self.Y_vv = nn.Parameter(torch.tensor(0.0))
        self.Z_ww = nn.Parameter(torch.tensor(0.0))
        self.K_pp = nn.Parameter(torch.tensor(0.0))
        self.M_qq = nn.Parameter(torch.tensor(0.0))
        self.N_rr = nn.Parameter(torch.tensor(0.0))

        # --------------------------
        # (6) Restoring / Buoyancy
        # --------------------------
        self.B   = nn.Parameter(torch.tensor(0.0))
        self.z_b = nn.Parameter(torch.tensor(-0.0))

        # --------------------------
        # (7) Constant Mass
        # --------------------------
        self.m_val = float(m_val)

        self.Crb = None
        self.Ca  = None
        self.C   = None
        self.D   = None

    # --- NEW helper ---
    def _get_inertias(self, device, dtype=torch.float32):
        """
        Returns (Ixx, Iyy, Izz) as tensors on the requested device/dtype.
        Uses learned params if enabled, otherwise CAD buffers.
        """
        if self.learn_inertia:
            Ixx = self.I_xx.to(device=device, dtype=dtype)
            Iyy = self.I_yy.to(device=device, dtype=dtype)
            Izz = self.I_zz.to(device=device, dtype=dtype)
        else:
            Ixx = self.Ixx_cad.to(device=device, dtype=dtype)
            Iyy = self.Iyy_cad.to(device=device, dtype=dtype)
            Izz = self.Izz_cad.to(device=device, dtype=dtype)
        return Ixx, Iyy, Izz

    # -------------------------------------------------------------------------
    # MASS MATRIX
    # -------------------------------------------------------------------------
    def build_mass_matrix(self) -> torch.Tensor:
        """
        M = M_RB + M_A (6x6)
        """
        # Added mass (note your convention: Ma = -diag([X_dot_u, ...]))
        Ma = -torch.diag(torch.stack([
            self.X_dot_u, self.Y_dot_v, self.Z_dot_w,
            self.K_dot_p, self.M_dot_q, self.N_dot_r
        ]))

        m  = torch.as_tensor(self.m_val, dtype=torch.float32, device=Ma.device)
        xg = self.x_g
        yg = self.y_g
        zg = self.z_g

        # NEW: get inertias (learned or CAD)
        Ixx, Iyy, Izz = self._get_inertias(device=Ma.device, dtype=torch.float32)

        Mrb = torch.zeros((6, 6), dtype=torch.float32, device=Ma.device)

        Mrb[0, 0] = m
        Mrb[1, 1] = m
        Mrb[2, 2] = m

        Mrb[0, 4] =  m * zg
        Mrb[0, 5] = -m * yg
        Mrb[1, 3] = -m * zg
        Mrb[1, 5] =  m * xg
        Mrb[2, 3] =  m * yg
        Mrb[2, 4] = -m * xg

        Mrb[3, 1] = -m * zg
        Mrb[3, 2] =  m * yg
        Mrb[3, 3] = Ixx

        Mrb[4, 0] =  m * zg
        Mrb[4, 2] = -m * xg
        Mrb[4, 4] = Iyy

        Mrb[5, 0] = -m * yg
        Mrb[5, 1] =  m * xg
        Mrb[5, 5] = Izz

        return Mrb + Ma

    # -------------------------------------------------------------------------
    # CORIOLIS: C = C_RB + C_A (with toggles)
    # -------------------------------------------------------------------------
    def build_coriolis(self, nu: torch.Tensor) -> torch.Tensor:
        """
        Returns C_total with flags controlling inclusion of Crb and Ca.
        nu: (B,6)
        """
        u, v, w, p, q, r = [nu[:, i] for i in range(6)]
        Bsz = nu.shape[0]

        m = torch.as_tensor(self.m_val, dtype=torch.float32, device=nu.device)

        Ixx, Iyy, Izz = self._get_inertias(device=nu.device, dtype=torch.float32)

        xg = self.x_g
        yg = self.y_g
        zg = self.z_g

        # --- Build Crb ---
        Crb = torch.zeros((Bsz, 6, 6), dtype=torch.float32, device=nu.device)

        Crb[:, 0, 1] = -m * r
        Crb[:, 0, 2] =  m * q
        Crb[:, 0, 3] =  m * (q * yg + r * zg)
        Crb[:, 0, 4] = -m * (q * xg)
        Crb[:, 0, 5] = -m * (r * xg)

        Crb[:, 1, 0] =  m * r
        Crb[:, 1, 2] = -m * p
        Crb[:, 1, 3] = -m * (p * yg)
        Crb[:, 1, 4] =  m * (p * xg + r * zg)
        Crb[:, 1, 5] = -m * (r * yg)

        Crb[:, 2, 0] = -m * q
        Crb[:, 2, 1] =  m * p
        Crb[:, 2, 3] = -m * (p * zg)
        Crb[:, 2, 4] = -m * (q * zg)
        Crb[:, 2, 5] =  m * (p * xg + q * yg)

        Crb[:, 3, 0] = -m * (q * yg + r * zg)
        Crb[:, 3, 1] =  m * (p * yg)
        Crb[:, 3, 2] =  m * (p * zg)
        Crb[:, 3, 4] =  Izz * r
        Crb[:, 3, 5] = -Iyy * q

        Crb[:, 4, 0] =  m * (q * xg)
        Crb[:, 4, 1] = -m * (p * xg + r * zg)
        Crb[:, 4, 2] =  m * (q * zg)
        Crb[:, 4, 3] = -Izz * r
        Crb[:, 4, 5] =  Ixx * p

        Crb[:, 5, 0] =  m * (r * xg)
        Crb[:, 5, 1] =  m * (r * yg)
        Crb[:, 5, 2] = -m * (p * xg + q * yg)
        Crb[:, 5, 3] =  Iyy * q
        Crb[:, 5, 4] = -Ixx * p

        # --- Build Ca (optional) ---
        Ca = torch.zeros_like(Crb)
        if self.use_Ca:
            a1 = self.X_dot_u * u
            a2 = self.Y_dot_v * v
            a3 = self.Z_dot_w * w
            b1 = self.K_dot_p * p
            b2 = self.M_dot_q * q
            b3 = self.N_dot_r * r

            Ca[:, 0, 4] = -a3
            Ca[:, 0, 5] =  a2
            Ca[:, 1, 3] =  a3
            Ca[:, 1, 5] = -a1
            Ca[:, 2, 3] = -a2
            Ca[:, 2, 4] =  a1

            Ca[:, 3, 1] = -a3
            Ca[:, 3, 2] =  a2
            Ca[:, 3, 4] = -b3
            Ca[:, 3, 5] =  b2

            Ca[:, 4, 0] =  a3
            Ca[:, 4, 2] = -a1
            Ca[:, 4, 3] =  b3
            Ca[:, 4, 5] = -b1

            Ca[:, 5, 0] = -a2
            Ca[:, 5, 1] =  a1
            Ca[:, 5, 3] = -b2
            Ca[:, 5, 4] =  b1

        # --- Combine per flags ---
        C_total = torch.zeros_like(Crb)
        if self.use_Crb:
            C_total = C_total + Crb
        if self.use_Ca:
            C_total = C_total + Ca

        # Store for debug
        self.Crb = Crb
        self.Ca  = Ca
        self.C   = C_total

        return C_total

    # -------------------------------------------------------------------------
    # DAMPING
    # -------------------------------------------------------------------------
    def build_damping(self, nu: torch.Tensor) -> torch.Tensor:
        """
        D(ν) = -diag(lin + quad*|nu|)  (batch,6,6)
        """
        lin_params = -torch.exp(torch.stack([self.X_u, self.Y_v, self.Z_w, self.K_p, self.M_q, self.N_r]))
        quad_params = torch.stack([self.X_uu, self.Y_vv, self.Z_ww, self.K_pp, self.M_qq, self.N_rr])

        diag_entries = lin_params.unsqueeze(0) + quad_params.unsqueeze(0) * torch.abs(nu)
        D = -1.0 * torch.diag_embed(diag_entries)

        self.D = D
        return D

    # -------------------------------------------------------------------------
    # RESTORING FORCE
    # -------------------------------------------------------------------------
    def build_restoring_force(self, eta: torch.Tensor) -> torch.Tensor:
        """
        g(eta) in NED (z down)
        eta: (B,6) [x,y,z,phi,theta,psi]
        """
        phi   = eta[:, 3]
        theta = eta[:, 4]

        g0 = 9.8
        m = torch.as_tensor(self.m_val, dtype=torch.float32, device=eta.device)
        W = m * g0
        B = self.B.to(dtype=torch.float32, device=eta.device)

        x_G, y_G, z_G = self.x_g, self.y_g, self.z_g
        x_B = torch.zeros(1, device=eta.device)
        y_B = torch.zeros(1, device=eta.device)
        z_B = self.z_b.to(dtype=torch.float32, device=eta.device)

        WB    = W - B
        xW_xB = x_G * W - x_B * B
        yW_yB = y_G * W - y_B * B
        zW_zB = z_G * W - z_B * B

        tau_g = torch.stack([
            WB * torch.sin(theta),
            -WB * torch.cos(theta) * torch.sin(phi),
            -WB * torch.cos(theta) * torch.cos(phi),
            -yW_yB * torch.cos(theta) * torch.cos(phi) + zW_zB * torch.cos(theta) * torch.sin(phi),
            zW_zB * torch.sin(theta) + xW_xB * torch.cos(theta) * torch.cos(phi),
            -xW_xB * torch.cos(theta) * torch.sin(phi) - yW_yB * torch.sin(theta)
        ], dim=1)

        return tau_g

    # -------------------------------------------------------------------------
    # FORWARD (inverse dynamics prediction)
    # -------------------------------------------------------------------------
    def forward(self, nu_dot: torch.Tensor, nu: torch.Tensor, eta: torch.Tensor) -> torch.Tensor:
        """
        nu_dot: (B,6)
        nu:     (B,6)
        eta:    (B,6)
        returns tau: (B,6)
        """
        # Mass
        M = self.build_mass_matrix()                 # (6,6)
        tau_M = torch.matmul(nu_dot, M.T)            # (B,6)

        # Coriolis (optional pieces inside)
        C = self.build_coriolis(nu)                  # (B,6,6)
        tau_C = torch.einsum("bij,bj->bi", C, nu)    # (B,6)

        # Damping
        if self.use_D:
            D = self.build_damping(nu)               # (B,6,6)
            tau_D = torch.einsum("bij,bj->bi", D, nu)
        else:
            tau_D = torch.zeros_like(tau_M)

        # Restoring
        if self.use_g:
            tau_g = self.build_restoring_force(eta)  # (B,6)
        else:
            tau_g = torch.zeros_like(tau_M)

        return tau_M + tau_C + tau_D + tau_g

    # -------------------------------------------------------------------------
    # REGULARIZATION HELPERS (unchanged)
    # -------------------------------------------------------------------------
    def L2reg(self, selected_names, prior_means):
        loss = 0.0
        for name, param in self.named_parameters():
            if name in selected_names:
                mu = prior_means[name].to(param.device)
                loss += torch.sum((param - mu) ** 2)
        return loss

    def L1reg(self, param_names):
        l1reg_sum = 0.0
        for name, param in self.named_parameters():
            if name in param_names:
                l1reg_sum += torch.sum(torch.abs(param))
        return l1reg_sum


# Define the Defender ROV Dynamics using the Fossen Model - SEPERATE ADDED MASS TERMS IN Ca



In [8]:



class ROVDynamicsModel(nn.Module):
    def __init__(self):
        super().__init__()

        # === (1) Added-Mass Coefficients (must be negative) ===
        self.X_dot_u = nn.Parameter(torch.tensor(-0.0))
        self.Y_dot_v = nn.Parameter(torch.tensor(-0.0))
        self.Z_dot_w = nn.Parameter(torch.tensor(-0.0))
        self.K_dot_p = nn.Parameter(torch.tensor(-0.0))
        self.M_dot_q = nn.Parameter(torch.tensor(-0.0))
        self.N_dot_r = nn.Parameter(torch.tensor(-0.0))

        # === (2) Rigid-Body Moments of Inertia ===
        self.I_xx = nn.Parameter(torch.tensor(0.0))
        self.I_yy = nn.Parameter(torch.tensor(0.0))
        self.I_zz = nn.Parameter(torch.tensor(0.0))

        # === (3) Center of Gravity Offsets ===
        self.x_g = nn.Parameter(torch.tensor(0.0))
        self.y_g = nn.Parameter(torch.tensor(0.0))
        self.z_g = nn.Parameter(torch.tensor(0.0))

        # === (4) Linear Damping Coefficients (must be negative) ===
        self.X_u = nn.Parameter(torch.tensor(0.0))
        self.Y_v = nn.Parameter(torch.tensor(0.0))
        self.Z_w = nn.Parameter(torch.tensor(0.0))
        self.K_p = nn.Parameter(torch.tensor(0.0))
        self.M_q = nn.Parameter(torch.tensor(0.0))
        self.N_r = nn.Parameter(torch.tensor(0.0))

        # === (5) Quadratic Damping Coefficients (must be negative) ===
        self.X_uu = nn.Parameter(torch.tensor(0.0))
        self.Y_vv = nn.Parameter(torch.tensor(0.0))
        self.Z_ww = nn.Parameter(torch.tensor(0.0))
        self.K_pp = nn.Parameter(torch.tensor(0.0))
        self.M_qq = nn.Parameter(torch.tensor(0.0))
        self.N_rr = nn.Parameter(torch.tensor(0.0))

                # === (NEW) Hydrodynamic Coriolis-only coupling parameters ===
        # These capture non-potential-flow Munk-like terms, vortex-turning, etc.
        # They affect ONLY C_A, not M_A.

        # === (NEW) Added-mass Coriolis-only terms ===
        # These replace X_dot_u, Y_dot_v, ... inside C_A only.
        self.Ca_X_dot_u = nn.Parameter(torch.tensor(0.0))
        self.Ca_Y_dot_v = nn.Parameter(torch.tensor(0.0))
        self.Ca_Z_dot_w = nn.Parameter(torch.tensor(0.0))

        self.Ca_K_dot_p = nn.Parameter(torch.tensor(0.0))
        self.Ca_M_dot_q = nn.Parameter(torch.tensor(0.0))
        self.Ca_N_dot_r = nn.Parameter(torch.tensor(0.0))



        # === (6) Restoring Force / Buoyancy Parameters ===
        # Positive z_b (NWU) means CB above CG → statically stable
        self.B = nn.Parameter(torch.tensor(0.0))  # ~169 N, neutral buoyancy
        self.z_b = nn.Parameter(torch.tensor(-0.0))        # CB above CG (positive upward in NWU)

        # === (7) Constant Physical Mass (non-learnable) ===
        # self.m_val = 17.2  # [kg], fixed vehicle mass
        self.m_val = 23.8 # [kg] new vehicle mass with weights added


    def build_mass_matrix(self):
        """
        Build the full 6×6 mass matrix M = M_RB + M_A (Fossen Eq. 3.49).
        Uses NED convention (z down).
        Fully differentiable (no torch.tensor() breaks).
        """

        Ma = -torch.diag(torch.stack([
            self.X_dot_u, self.Y_dot_v, self.Z_dot_w, self.K_dot_p, self.M_dot_q, self.N_dot_r
        ]))

        # --- Rigid-body mass matrix (M_RB) ---
        m = torch.as_tensor(self.m_val, dtype=torch.float32, device=Ma.device)
        xg = self.x_g
        yg = self.y_g
        zg = self.z_g

        # build element-wise so autograd stays live
        Mrb = torch.zeros((6, 6), dtype=torch.float32, device=Ma.device)

        Mrb[0, 0] = m
        Mrb[1, 1] = m
        Mrb[2, 2] = m
        Mrb[0, 4] =  m * zg
        Mrb[0, 5] = -m * yg
        Mrb[1, 3] = -m * zg
        Mrb[1, 5] =  m * xg
        Mrb[2, 3] =  m * yg
        Mrb[2, 4] = -m * xg
        Mrb[3, 1] = -m * zg
        Mrb[3, 2] =  m * yg
        Mrb[3, 3] = self.I_xx
        Mrb[4, 0] =  m * zg
        Mrb[4, 2] = -m * xg
        Mrb[4, 4] = self.I_yy
        Mrb[5, 0] = -m * yg
        Mrb[5, 1] =  m * xg
        Mrb[5, 5] = self.I_zz

        # --- Total mass matrix ---
        M = Mrb + Ma
        return M



    def build_coriolis(self, nu):
        """
        Build the full 6×6 Coriolis and centripetal matrix:
            C(ν) = C_RB(ν) + C_A(ν)
        using Fossen Eqs. (3.53) and (6.46), assuming CG = 0 (NED convention).
        All parameters remain gradient-connected.
        """
        u, v, w, p, q, r = [nu[:, i] for i in range(6)]
        B = nu.shape[0]

        # === Constants and inertias ===
        m = torch.as_tensor(self.m_val, dtype=torch.float32, device=nu.device)
        Ixx, Iyy, Izz = self.I_xx, self.I_yy, self.I_zz

        # === Build C_RB (rigid-body) ===
        Crb = torch.zeros((B, 6, 6), dtype=torch.float32, device=nu.device)

           # shorthand
        xg = self.x_g
        yg = self.y_g
        zg = self.z_g

        Crb[:, 0, 0] = 0.0
        Crb[:, 0, 1] = -m * r
        Crb[:, 0, 2] =  m * q
        Crb[:, 0, 3] =  m * (q * yg + r * zg)
        Crb[:, 0, 4] = -m * (q * xg)
        Crb[:, 0, 5] = -m * (r * xg)

        Crb[:, 1, 0] =  m * r
        Crb[:, 1, 1] = 0.0
        Crb[:, 1, 2] = -m * p
        Crb[:, 1, 3] = -m * (p * yg)
        Crb[:, 1, 4] =  m * (p * xg + r * zg)
        Crb[:, 1, 5] = -m * (r * yg)

        Crb[:, 2, 0] = -m * q
        Crb[:, 2, 1] =  m * p
        Crb[:, 2, 2] = 0.0
        Crb[:, 2, 3] = -m * (p * zg)
        Crb[:, 2, 4] = -m * (q * zg)
        Crb[:, 2, 5] =  m * (p * xg + q * yg)

        Crb[:, 3, 0] = -m * (q * yg + r * zg)
        Crb[:, 3, 1] =  m * (p * yg)
        Crb[:, 3, 2] =  m * (p * zg)
        Crb[:, 3, 3] = 0.0
        Crb[:, 3, 4] =  Izz * r
        Crb[:, 3, 5] = -Iyy * q

        Crb[:, 4, 0] =  m * (q * xg)
        Crb[:, 4, 1] = -m * (p * xg + r * zg)
        Crb[:, 4, 2] =  m * (q * zg)
        Crb[:, 4, 3] = -Izz * r
        Crb[:, 4, 4] = 0.0
        Crb[:, 4, 5] =  Ixx * p

        Crb[:, 5, 0] =  m * (r * xg)
        Crb[:, 5, 1] =  m * (r * yg)
        Crb[:, 5, 2] = -m * (p * xg + q * yg)
        Crb[:, 5, 3] =  Iyy * q      # (2,0) =  w_y
        Crb[:, 5, 4] = -Ixx * p      # (2,1) = -w_x
        Crb[:, 5, 5] = 0.0

        # === Build C_A (added-mass) ===
        a1 = self.Ca_X_dot_u * u
        a2 = self.Ca_Y_dot_v * v
        a3 = self.Ca_Z_dot_w * w

        b1 = self.Ca_K_dot_p * p
        b2 = self.Ca_M_dot_q * q
        b3 = self.Ca_N_dot_r * r


        Ca = torch.zeros_like(Crb)

        Ca[:, 0, 4] = -a3
        Ca[:, 0, 5] =  a2
        Ca[:, 1, 3] =  a3
        Ca[:, 1, 5] = -a1
        Ca[:, 2, 3] = -a2
        Ca[:, 2, 4] =  a1

        Ca[:, 3, 1] = -a3
        Ca[:, 3, 2] =  a2
        Ca[:, 3, 4] = -b3
        Ca[:, 3, 5] =  b2

        Ca[:, 4, 0] =  a3
        Ca[:, 4, 2] = -a1
        Ca[:, 4, 3] =  b3
        Ca[:, 4, 5] = -b1

        Ca[:, 5, 0] = -a2
        Ca[:, 5, 1] =  a1
        Ca[:, 5, 3] = -b2
        Ca[:, 5, 4] =  b1

        # === Total Coriolis ===
        C_total = Crb + Ca
        # Optional: enforce skew-symmetry for energy conservation
        # C_total = 0.5 * (C_total - C_total.transpose(1, 2))

        self.Crb = Crb
        self.Ca  = Ca
        self.C   = C_total
        return C_total


    def build_damping(self, nu):
        """
        Damping matrix D(ν) with negative coefficients AND a leading -1,
        exactly like your simulator:

            D1 = -diag([X_u, Y_v, Z_w, K_p, M_q, N_r])
            D2 = -diag([X_uu|u|, Y_vv|v|, Z_ww|w|, K_pp|p|, M_qq|q|, N_rr|r|])
            D  = D1 + D2
            τ_D = D(ν) ν

        NOTE:
            - Coefficients (X_u, X_uu, ...) are expected to be NEGATIVE numbers.
            - This implementation preserves gradient flow for training.
        """
        u = nu[:, 0]
        v = nu[:, 1]
        w = nu[:, 2]
        p = nu[:, 3]
        q = nu[:, 4]
        r = nu[:, 5]

        B = nu.shape[0]

        # Stack parameter tensors directly (no torch.tensor()!)
        lin_params = torch.stack([
            self.X_u, self.Y_v, self.Z_w,
            self.K_p, self.M_q, self.N_r
        ])

        quad_params = torch.stack([
            self.X_uu, self.Y_vv, self.Z_ww,
            self.K_pp, self.M_qq, self.N_rr
        ])

        # Compute per-batch diagonal entries
        diag_entries = lin_params.unsqueeze(0) + quad_params.unsqueeze(0) * torch.abs(nu)
        D = -1.0 * torch.diag_embed(diag_entries)   # Apply leading negative, same as your sim

        # --- Optional sanity check: dissipativity ---
        with torch.no_grad():
            dn = torch.einsum('bi,bij,bj->b', nu, D, nu)
            if (dn > 1e-9).any():
                # You can uncomment if you want a runtime alert
                # print(f"[warn] damping energy injection detected: {(dn>0).sum().item()} samples")
                pass

        self.D = D
        return D


    def build_restoring_force(self, eta):
        """
        Compute restoring forces and moments τ_g(η)
        using Fossen Eq. (3.77) in the NED convention (z down).

        Parameters
        ----------
        eta : torch.Tensor, shape (batch, 6)
            Vehicle pose vector [x, y, z, φ, θ, ψ]ᵀ in radians.

        Returns
        -------
        tau_g : torch.Tensor, shape (batch, 6)
            Restoring force/moment vector in body frame.
        """
        phi = eta[:, 3]
        theta = eta[:, 4]

        # --- Physical constants ---
        g = 9.8
        m = torch.as_tensor(self.m_val, dtype=torch.float32, device=eta.device)  # constant (non-trainable)
        W = m * g
        B = self.B.to(dtype=torch.float32, device=eta.device)

        # === Centers of gravity and buoyancy ===
        x_G, y_G, z_G = self.x_g, self.y_g, self.z_g
        # buoyancy center (x_B, y_B fixed at 0, z_B trainable)
        x_B = torch.zeros(1, device=eta.device)
        y_B = torch.zeros(1, device=eta.device)
        z_B = self.z_b.to(dtype=torch.float32, device=eta.device)

        # --- Precompute weight–buoyancy differences ---
        WB    = W - B
        xW_xB = x_G * W - x_B * B
        yW_yB = y_G * W - y_B * B
        zW_zB = z_G * W - z_B * B

        # --- Restoring forces (batch × 6) ---
        tau_g = torch.stack([
            WB * torch.sin(theta),
            -WB * torch.cos(theta) * torch.sin(phi),
            -WB * torch.cos(theta) * torch.cos(phi),
            -yW_yB * torch.cos(theta) * torch.cos(phi) + zW_zB * torch.cos(theta) * torch.sin(phi),
            zW_zB * torch.sin(theta) + xW_xB * torch.cos(theta) * torch.cos(phi),
            -xW_xB * torch.cos(theta) * torch.sin(phi) - yW_yB * torch.sin(theta)
        ], dim=1)

        return tau_g





    def forward(self, nu_dot, nu, eta):
    # === Mass force: M * nu_dot ===
        M = self.build_mass_matrix()                  # shape: (6, 6), same for all samples
        tau_M = torch.matmul(nu_dot, M.T)             # shape: (batch, 6)

    # === Coriolis force: C(nu) * nu ===
        C = self.build_coriolis(nu)            # shape: (batch, 6, 6)
        tau_C = torch.einsum("bij,bj->bi", C, nu)     # shape: (batch, 6)

    # === Damping force: nonlinear diagonal ===
        D = self.build_damping(nu)
        tau_D = torch.einsum("bij,bj->bi", D, nu)       # shape: (batch, 6)

    # === Restoring force: g(eta) ===
        tau_g = self.build_restoring_force(eta)       # shape: (batch, 6)

    # === Total force and moment ===
        tau = tau_M + tau_C + tau_D + tau_g           # shape: (batch, 6)

        return tau

    def L2reg(self, selected_names, prior_means):
        loss = 0.0
        for name, param in self.named_parameters():
            if name in selected_names:
                mu = prior_means[name].to(param.device)
                loss += torch.sum((param - mu) ** 2)
        return loss


    def L1reg(self, param_names):
        l1reg_sum = 0.0
        for name, param in self.named_parameters():
            if name in param_names:
                l1reg_sum += torch.sum(torch.abs(param))
        return l1reg_sum



#  Dynamics Node with NO CA to EXCLUDE MUNK MOMENTS

In [15]:
class ROVDynamicsModel(nn.Module):
    def __init__(self):
        super().__init__()

        # === (1) Added-Mass Coefficients (negative) ===
        self.X_dot_u = nn.Parameter(torch.tensor(-0.0))
        self.Y_dot_v = nn.Parameter(torch.tensor(-0.0))
        self.Z_dot_w = nn.Parameter(torch.tensor(-0.0))
        self.K_dot_p = nn.Parameter(torch.tensor(-0.0))
        self.M_dot_q = nn.Parameter(torch.tensor(-0.0))
        self.N_dot_r = nn.Parameter(torch.tensor(-0.0))

        # === (2) Rigid-Body Moments of Inertia ===
        self.I_xx = nn.Parameter(torch.tensor(0.0))
        self.I_yy = nn.Parameter(torch.tensor(0.0))
        self.I_zz = nn.Parameter(torch.tensor(0.0))

        # === (3) Center of Gravity Offsets ===
        self.x_g = nn.Parameter(torch.tensor(0.0))
        self.y_g = nn.Parameter(torch.tensor(0.0))
        self.z_g = nn.Parameter(torch.tensor(0.0))

        # === (4) Linear Damping Coefficients (negative) ===
        self.X_u = nn.Parameter(torch.tensor(0.0))
        self.Y_v = nn.Parameter(torch.tensor(0.0))
        self.Z_w = nn.Parameter(torch.tensor(0.0))
        self.K_p = nn.Parameter(torch.tensor(0.0))
        self.M_q = nn.Parameter(torch.tensor(0.0))
        self.N_r = nn.Parameter(torch.tensor(0.0))

        # === (5) Quadratic Damping Coefficients (negative) ===
        self.X_uu = nn.Parameter(torch.tensor(0.0))
        self.Y_vv = nn.Parameter(torch.tensor(0.0))
        self.Z_ww = nn.Parameter(torch.tensor(0.0))
        self.K_pp = nn.Parameter(torch.tensor(0.0))
        self.M_qq = nn.Parameter(torch.tensor(0.0))
        self.N_rr = nn.Parameter(torch.tensor(0.0))

        # === (6) Restoring Force / Buoyancy ===
        self.B   = nn.Parameter(torch.tensor(0.0))
        self.z_b = nn.Parameter(torch.tensor(-0.0))

        # === (7) Constant Vehicle Mass ===
        self.m_val = 23.8  # [kg]

    # ------------------------------------------------------------------
    # Mass matrix: M = M_RB + M_A
    # ------------------------------------------------------------------
    def build_mass_matrix(self):
        Ma = -torch.diag(torch.stack([
            self.X_dot_u, self.Y_dot_v, self.Z_dot_w,
            self.K_dot_p, self.M_dot_q, self.N_dot_r
        ]))

        m  = torch.as_tensor(self.m_val, dtype=torch.float32, device=Ma.device)
        xg, yg, zg = self.x_g, self.y_g, self.z_g

        Mrb = torch.zeros((6, 6), dtype=torch.float32, device=Ma.device)

        Mrb[0, 0] = m
        Mrb[1, 1] = m
        Mrb[2, 2] = m

        Mrb[0, 4] =  m * zg
        Mrb[0, 5] = -m * yg
        Mrb[1, 3] = -m * zg
        Mrb[1, 5] =  m * xg
        Mrb[2, 3] =  m * yg
        Mrb[2, 4] = -m * xg

        Mrb[3, 1] = -m * zg
        Mrb[3, 2] =  m * yg
        Mrb[3, 3] = self.I_xx

        Mrb[4, 0] =  m * zg
        Mrb[4, 2] = -m * xg
        Mrb[4, 4] = self.I_yy

        Mrb[5, 0] = -m * yg
        Mrb[5, 1] =  m * xg
        Mrb[5, 5] = self.I_zz

        return Mrb + Ma

    # ------------------------------------------------------------------
    # Coriolis matrix: ONLY rigid-body Coriolis (C_A removed)
    # ------------------------------------------------------------------
    def build_coriolis(self, nu):
        u, v, w, p, q, r = [nu[:, i] for i in range(6)]
        B = nu.shape[0]

        m  = torch.as_tensor(self.m_val, dtype=torch.float32, device=nu.device)
        Ixx, Iyy, Izz = self.I_xx, self.I_yy, self.I_zz
        xg, yg, zg = self.x_g, self.y_g, self.z_g

        Crb = torch.zeros((B, 6, 6), dtype=torch.float32, device=nu.device)

        Crb[:, 0, 1] = -m * r
        Crb[:, 0, 2] =  m * q
        Crb[:, 0, 3] =  m * (q * yg + r * zg)
        Crb[:, 0, 4] = -m * (q * xg)
        Crb[:, 0, 5] = -m * (r * xg)

        Crb[:, 1, 0] =  m * r
        Crb[:, 1, 2] = -m * p
        Crb[:, 1, 3] = -m * (p * yg)
        Crb[:, 1, 4] =  m * (p * xg + r * zg)
        Crb[:, 1, 5] = -m * (r * yg)

        Crb[:, 2, 0] = -m * q
        Crb[:, 2, 1] =  m * p
        Crb[:, 2, 3] = -m * (p * zg)
        Crb[:, 2, 4] = -m * (q * zg)
        Crb[:, 2, 5] =  m * (p * xg + q * yg)

        Crb[:, 3, 0] = -m * (q * yg + r * zg)
        Crb[:, 3, 1] =  m * (p * yg)
        Crb[:, 3, 2] =  m * (p * zg)
        Crb[:, 3, 4] =  Izz * r
        Crb[:, 3, 5] = -Iyy * q

        Crb[:, 4, 0] =  m * (q * xg)
        Crb[:, 4, 1] = -m * (p * xg + r * zg)
        Crb[:, 4, 2] =  m * (q * zg)
        Crb[:, 4, 3] = -Izz * r
        Crb[:, 4, 5] =  Ixx * p

        Crb[:, 5, 0] =  m * (r * xg)
        Crb[:, 5, 1] =  m * (r * yg)
        Crb[:, 5, 2] = -m * (p * xg + q * yg)
        Crb[:, 5, 3] =  Iyy * q
        Crb[:, 5, 4] = -Ixx * p

        self.Crb = Crb
        return Crb

    # ------------------------------------------------------------------
    # Damping: diagonal linear + quadratic
    # ------------------------------------------------------------------
    def build_damping(self, nu):
        lin = torch.stack([self.X_u, self.Y_v, self.Z_w,
                           self.K_p, self.M_q, self.N_r])
        quad = torch.stack([self.X_uu, self.Y_vv, self.Z_ww,
                            self.K_pp, self.M_qq, self.N_rr])

        diag = lin.unsqueeze(0) + quad.unsqueeze(0) * torch.abs(nu)
        D = -torch.diag_embed(diag)

        self.D = D
        return D

    # ------------------------------------------------------------------
    # Restoring forces
    # ------------------------------------------------------------------
    def build_restoring_force(self, eta):
        phi   = eta[:, 3]
        theta = eta[:, 4]

        g = 9.8
        m = torch.as_tensor(self.m_val, dtype=torch.float32, device=eta.device)
        W = m * g
        B = self.B

        xG, yG, zG = self.x_g, self.y_g, self.z_g
        zB = self.z_b

        WB = W - B

        return torch.stack([
            WB * torch.sin(theta),
            -WB * torch.cos(theta) * torch.sin(phi),
            -WB * torch.cos(theta) * torch.cos(phi),
            (zG * W - zB * B) * torch.cos(theta) * torch.sin(phi),
            (zG * W - zB * B) * torch.sin(theta),
            torch.zeros_like(theta)
        ], dim=1)

    # ------------------------------------------------------------------
    # Forward model
    # ------------------------------------------------------------------
    def forward(self, nu_dot, nu, eta):
        M = self.build_mass_matrix()
        tau_M = nu_dot @ M.T

        C = self.build_coriolis(nu)
        tau_C = torch.einsum("bij,bj->bi", C, nu)

        D = self.build_damping(nu)
        tau_D = torch.einsum("bij,bj->bi", D, nu)

        tau_g = self.build_restoring_force(eta)

        return tau_M + tau_C + tau_D + tau_g


In [23]:
def nll_tau(tau_true, tau_pred, std):
    return torch.square(tau_true - tau_pred) / (2.0 * std**2) + torch.log(std)
def mse(tau_true, tau_pred):
    return torch.mean((tau_true - tau_pred) ** 2)

# Defender MLE Regression

In [121]:
model = ROVDynamicsModel()
optimizer = torch.optim.Adam(model.parameters(), lr=1.5)

ac_cuda  = nu_dot_train
v_cuda   = nu_train
eta_cuda = eta_train

for i in range(n_epochs):
    optimizer.zero_grad()  # ✅ clear grads before forward

    # === Forward pass ===
    tau_pred = model(ac_cuda, v_cuda, eta_cuda)
    mse_loss = F.mse_loss(tau_pred, y_train)

    # total loss
    loss = mse_loss

    # === Backward pass ===

    optimizer.zero_grad()
    loss.backward()

    # # === Debug: check damping parameter gradients ===
    # print(f"\n[Epoch {i}] loss={loss.item():.4e}")
    # for name, param in model.named_parameters():
    #     if any(k in name for k in ["_u", "_v", "_w"]):  # linear damping params
    #         grad = param.grad
    #         print(f"{name:8s} grad:", None if grad is None else grad.item())

    # === Update step ===
    optimizer.step()

    if verbose_option:
        print(f"Epoch {i} | Loss: {loss.item():.6f}")


    # with torch.no_grad():
    #     print("\n============================")
    #     print("🚢 ROV DYNAMICS DIAGNOSTIC")
    #     print("============================4  1.764625e+09    0.0    0.0 -0.06150    0.0    0.0    0.0  0.0  0.0   ")
    # #
    #     # --- Snapshot of key parameters ---
    #     for name, param in model.named_parameters():
    #         if any(key in name for key in ["dot", "u", "v", "w", "p", "q", "r", "B", "z_b"]):
    #             print(f"{name:12s}: {param.item():+10.4f}")
    #
        # --- Select one representative sample ---
        # nu     = nu_train[0:1]      # body-frame velocity [u,v,w,p,q,r]
        # nu_dot = nu_dot_train[0:1]  # body-frame acceleration
        # eta    = eta_train[0:1]     # pose [x,y,z,φ,θ,ψ]
    #
    #     # --- Compute model components ---
    #     M = model.build_mass_matrix()        # (6×6)
    #     C = model.build_coriolis(nu)[0]      # (6×6)
    #     D = model.build_damping(nu)[0]       # (6×6)
    #     g = model.build_restoring_force(eta)[0]  # (6,)
    #
    #     # --- Individual contributions ---
    #     tau_M = torch.matmul(nu_dot, M.T)[0]         # (6,)
    #     tau_C = torch.matmul(C, nu[0])               # (6,)
    #     tau_D = torch.matmul(D, nu[0])               # (6,)
    #     tau_g = g                                    # (6,)
    #
        #tau_total = tau_M + tau_C + tau_D + tau_g
        # tau_total = tau_M + tau_D
    #
    #     # === Print Outputs ===
    #     print("\n=== MASS MATRIX M (kg, kg·m²) ===")
    #     print(M.cpu().numpy().round(3))
    #
    #     print("\n=== CORIOLIS MATRIX C (should be skew-symmetric, νᵀCν ≈ 0) ===")
    #     print(C.cpu().numpy().round(3))
    #
        # print("\n=== DAMPING MATRIX D (negative definite) ===")
        # print(D.cpu().numpy().round(3))
    #
    #     print("\n=== RESTORING FORCE VECTOR g(η) ===")
    #     print(g.cpu().numpy().round(3))
    #
    #     print("\n=== FORCE CONTRIBUTIONS (body frame) ===")
    #     print(f"τ_M (mass*accel): {tau_M.cpu().numpy().round(3)}")
    #     print(f"τ_C (Coriolis):   {tau_C.cpu().numpy().round(3)}")
    #     print(f"τ_D (Damping):    {tau_D.cpu().numpy().round(3)}")
    #     print(f"τ_g (Restoring):  {tau_g.cpu().numpy().round(3)}")
    #     print(f"τ_total:          {tau_total.cpu().numpy().round(3)}")
    # #
    #     # --- Energy consistency tests ---
    #     work_C = torch.einsum('bi,bij,bj->b', nu, model.build_coriolis(nu), nu)
    #     work_D = torch.einsum('bi,bij,bj->b', nu, model.build_damping(nu), nu)
    #
    #     print("\n=== ENERGY CHECKS ===")
    #     print(f"νᵀCν (should be ≈ 0): {work_C.item():+.3e}")
    #     print(f"νᵀDν (should be ≤ 0): {work_D.item():+.3e}")





Epoch 0 | Loss: 10194.978516
Epoch 1 | Loss: 56532.484375
Epoch 2 | Loss: 12847.778320
Epoch 3 | Loss: 19486.636719
Epoch 4 | Loss: 36234.324219
Epoch 5 | Loss: 29737.816406
Epoch 6 | Loss: 15119.754883
Epoch 7 | Loss: 8934.001953
Epoch 8 | Loss: 14188.087891
Epoch 9 | Loss: 21135.511719
Epoch 10 | Loss: 20965.228516
Epoch 11 | Loss: 14839.584961
Epoch 12 | Loss: 9160.810547
Epoch 13 | Loss: 8449.943359
Epoch 14 | Loss: 11825.848633
Epoch 15 | Loss: 14697.515625
Epoch 16 | Loss: 13937.482422
Epoch 17 | Loss: 10540.792969
Epoch 18 | Loss: 7724.387207
Epoch 19 | Loss: 7611.765137
Epoch 20 | Loss: 9486.547852
Epoch 21 | Loss: 10878.258789
Epoch 22 | Loss: 10239.518555
Epoch 23 | Loss: 8264.936523
Epoch 24 | Loss: 6808.686035
Epoch 25 | Loss: 6938.446289
Epoch 26 | Loss: 8024.258301
Epoch 27 | Loss: 8602.827148
Epoch 28 | Loss: 7976.013672
Epoch 29 | Loss: 6782.897949
Epoch 30 | Loss: 6135.323730
Epoch 31 | Loss: 6422.578125
Epoch 32 | Loss: 7015.986816
Epoch 33 | Loss: 7076.109863
Epoch 3

KeyboardInterrupt: 

In [122]:
print("\n=== Model Parameter Summary ===")
print(f"{'Parameter':<20} {'Value':>12}")
print("-" * 34)

for name, param in model.named_parameters():
    val = param.detach().cpu().numpy().item() if param.numel() == 1 else "array"
    print(f"{name:<20} {val:>12}")



=== Model Parameter Summary ===
Parameter                   Value
----------------------------------
X_dot_u              23.794597625732422
Y_dot_v              23.443878173828125
Z_dot_w              -79.58185577392578
K_dot_p              1.0586295127868652
M_dot_q              1.2803388833999634
N_dot_r              0.4110085070133209
x_g                  -0.009176691062748432
y_g                  -0.0008755732560530305
z_g                  0.020364120602607727
X_u                  -2.0403287410736084
Y_v                  -14.404219627380371
Z_w                  -35.66729736328125
K_p                  2.903256893157959
M_q                  2.715120553970337
N_r                  0.8657656311988831
X_uu                 -99.82315063476562
Y_vv                 -98.48116302490234
Z_ww                 -128.30606079101562
K_pp                 -5.371804714202881
M_qq                 1.6916825771331787
N_rr                 -7.195874214172363
B                    235.9845428466797
z_b      

In [11]:

model.eval()
with torch.no_grad():
    tau_pred = model(nu_dot_test, nu_test, eta_test)

    # ======================================================
    # 1. Active samples and active DOFs
    # ======================================================
    active_sample_mask = torch.any(torch.abs(y_test) > 1e-3, dim=1)
    max_force_per_axis, _ = torch.max(torch.abs(y_test[active_sample_mask]), dim=0)
    dof_mask = max_force_per_axis > 1e-3

    # ======================================================
    # 2. RMSE metrics
    # ======================================================
    mse  = F.mse_loss(tau_pred, y_test, reduction="mean")
    rmse = torch.sqrt(mse)

    per_axis_rmse = torch.sqrt(torch.mean((tau_pred - y_test) ** 2, dim=0))

    eps = 1e-6
    y_active = y_test[active_sample_mask]
    tau_active = tau_pred[active_sample_mask]
    mean_force_per_axis = torch.mean(torch.abs(y_active), dim=0)
    max_force_per_axis, _ = torch.max(torch.abs(y_active), dim=0)

    rel_rmse_mean = torch.full_like(per_axis_rmse, float("nan"))
    rel_rmse_max  = torch.full_like(per_axis_rmse, float("nan"))

    rel_rmse_mean[dof_mask] = 100 * per_axis_rmse[dof_mask] / (mean_force_per_axis[dof_mask] + eps)
    rel_rmse_max[dof_mask]  = 100 * per_axis_rmse[dof_mask] / (max_force_per_axis[dof_mask] + eps)

    active_rmse_mean = torch.mean(per_axis_rmse[dof_mask])

    # ======================================================
    # 3. Nicely formatted table
    # ======================================================
    dof_labels = ["X", "Y", "Z", "K", "M", "N"]
    df = pd.DataFrame({
        "DOF": dof_labels,
        "RMSE [N]": per_axis_rmse.cpu().numpy(),
        "Rel RMSE (mean) [%]": rel_rmse_mean.cpu().numpy(),
        "Rel RMSE (max) [%]": rel_rmse_max.cpu().numpy(),
        "Active?": dof_mask.cpu().numpy()
    })

    print("\n=== Test Set Error Metrics ===")
    print(f"Global MSE  : {mse.item():.6f} N²")
    print(f"Global RMSE : {rmse.item():.3f} N")
    print(f"Active-axis mean RMSE: {active_rmse_mean.item():.3f} N\n")

    # Pretty table (rounded and aligned)
    print(df.to_string(index=False, float_format=lambda x: f"{x:8.3f}"))



=== Test Set Error Metrics ===
Global MSE  : 0.001734 N²
Global RMSE : 0.042 N
Active-axis mean RMSE: 0.081 N

DOF  RMSE [N]  Rel RMSE (mean) [%]  Rel RMSE (max) [%]  Active?
  X     0.004                  NaN                 NaN    False
  Y     0.005                  NaN                 NaN    False
  Z     0.062                  NaN                 NaN    False
  K     0.000                  NaN                 NaN    False
  M     0.000                  NaN                 NaN    False
  N     0.081                0.958               0.510     True


## Defender MAP Regresssion

In [176]:
# ==== PRIOR MEANS FROM TANK TESTING ===


prior_means = {
    "X_dot_u": torch.tensor(-30.579),
    "Y_dot_v": torch.tensor(-30.938),
    "Z_dot_w": torch.tensor(-56.142),
    "K_dot_p": torch.tensor(-0.282),
    "M_dot_q": torch.tensor(-0.599),
    "N_dot_r": torch.tensor(-0.523),

    "I_xx": torch.tensor(0.393), #from CAD
    "I_yy": torch.tensor(1.302), #from CAD
    "I_zz": torch.tensor(1.429), #from CAD

    "x_g": torch.tensor(0.0),
    "y_g": torch.tensor(0.0),
    "z_g": torch.tensor(0.0),

    "X_u": torch.tensor(-2.00),
    "Y_v": torch.tensor(-2.00),
    "Z_w": torch.tensor(-2.00),
    "K_p": torch.tensor(-1.00),
    "M_q": torch.tensor(-1.00),
    "N_r": torch.tensor(-1.00),

    "X_uu": torch.tensor(-28.875),
    "Y_vv": torch.tensor(-95.716),
    "Z_ww": torch.tensor(-106.526),
    "K_pp": torch.tensor(-0.592),
    "M_qq": torch.tensor(-1.811),
    "N_rr": torch.tensor(-4.734),

    "B": torch.tensor(234.36),
    "z_b": torch.tensor(-0.02)
}


In [50]:
# ==== PRIOR MEANS FOR SIMULATION ===

prior_means = {
    "X_dot_u": torch.tensor(-18.0),
    "Y_dot_v": torch.tensor(-22.584),
    "Z_dot_w": torch.tensor(-22.3775),
    "K_dot_p": torch.tensor(-0.079),
    "M_dot_q": torch.tensor(-0.26),
    "N_dot_r": torch.tensor(-0.286),

    "I_xx": torch.tensor(1.0),
    "I_yy": torch.tensor(1.0),
    "I_zz": torch.tensor(1.0),

    "x_g": torch.tensor(0.0),
    "y_g": torch.tensor(0.0),
    "z_g": torch.tensor(0.0),

    "X_u": torch.tensor(-4.6),
    "Y_v": torch.tensor(-12.6),
    "Z_w": torch.tensor(-14.17),
    "K_p": torch.tensor(-1.5),
    "M_q": torch.tensor(-2.9),
    "N_r": torch.tensor(-0.101),

    "X_uu": torch.tensor(-51.8358),
    "Y_vv": torch.tensor(-102.01),
    "Z_ww": torch.tensor(-155.84),
    "K_pp": torch.tensor(-2.1),
    "M_qq": torch.tensor(-14.6),
    "N_rr": torch.tensor(-4.096),

    "B": torch.tensor(168.56),
    "z_b": torch.tensor(-0.05)
}


In [177]:
# ============================================================
# === MAP configuration ======================================
# ============================================================

# Regularization strengths
l1_coeff       = 1.0     # Laplace prior strength (sparsity) on linear damping
l2_tight_coeff = 1e-2    # tight Gaussian prior on inertias (CAD)
l2_loose_coeff = 4e-4    # loose Gaussian prior on hydros

# Parameter buckets
L2_TIGHT = ["I_xx", "I_yy", "I_zz","x_g","y_g","z_g"]  # only if learn_inertia=True

L2_LOOSE = [
    "X_dot_u","Y_dot_v","Z_dot_w","K_dot_p","M_dot_q","N_dot_r",
    "X_uu","Y_vv","Z_ww","K_pp","M_qq","N_rr",
    "B","z_b",
    # "x_g","y_g","z_g",   # <- remove
]

# L1: add CG offsets (sparse around 0)
L1_LINEAR = ["X_u","Y_v","Z_w","K_p","M_q","N_r"]




# ============================================================
# === Model + optimizer =====================================
# ============================================================

model = ROVDynamicsModel()
optimizer = optim.Adam(model.parameters(), lr=0.5)

device = next(model.parameters()).device
for k, v in prior_means.items():
    prior_means[k] = v.to(device).float()

# Only keep keys that exist in prior_means
all_params = set(prior_means.keys())
L2_TIGHT   = [p for p in L2_TIGHT   if p in all_params]
L2_LOOSE   = [p for p in L2_LOOSE   if p in all_params]
L1_LINEAR  = [p for p in L1_LINEAR  if p in all_params]

# Optional: print what you're regularizing
print("\n=== MAP regularization sets ===")
print(f"L2_TIGHT ({len(L2_TIGHT)}): {L2_TIGHT}")
print(f"L2_LOOSE ({len(L2_LOOSE)}): {L2_LOOSE}")
print(f"L1_LINEAR({len(L1_LINEAR)}): {L1_LINEAR}")
print("================================\n")



# ============================================================
# === Training loop ==========================================
# ============================================================
for epoch in range(n_epochs):

    tau_pred = model(nu_dot_train, nu_train, eta_train)

    # NLL (Gaussian noise -> MSE up to constant scale)
    nll_loss = F.mse_loss(tau_pred, y_train)

    # Priors / regularization
    l2_tight_term = model.L2reg(L2_TIGHT, prior_means)
    l2_loose_term = model.L2reg(L2_LOOSE, prior_means)
    l1_term       = model.L1reg(L1_LINEAR)

    loss = (
        nll_loss
        + l2_tight_coeff * l2_tight_term
        + l2_loose_coeff * l2_loose_term
        + l1_coeff * l1_term
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # if verbose_option and epoch % 50 == 0:
    #     print(
    #         f"Epoch {epoch:4d} | "
    #         f"Loss={loss.item():.4e} | "
    #         f"NLL={nll_loss.item():.4e} | "
    #         f"L2tight={l2_tight_term.item():.4e} | "
    #         f"L2loose={l2_loose_term.item():.4e} | "
    #         f"L1={l1_term.item():.4e}"
    #     )

    if verbose_option and epoch % 50 == 0:
        print(f"Epoch {epoch:4d} | Loss={loss.item():.6f}")





=== MAP regularization sets ===
L2_TIGHT (6): ['I_xx', 'I_yy', 'I_zz', 'x_g', 'y_g', 'z_g']
L2_LOOSE (14): ['X_dot_u', 'Y_dot_v', 'Z_dot_w', 'K_dot_p', 'M_dot_q', 'N_dot_r', 'X_uu', 'Y_vv', 'Z_ww', 'K_pp', 'M_qq', 'N_rr', 'B', 'z_b']
L1_LINEAR(6): ['X_u', 'Y_v', 'Z_w', 'K_p', 'M_q', 'N_r']

Epoch    0 | Loss=9180.824219
Epoch   50 | Loss=7250.588379
Epoch  100 | Loss=5705.369629
Epoch  150 | Loss=4436.227051
Epoch  200 | Loss=3401.809082
Epoch  250 | Loss=2569.821045
Epoch  300 | Loss=1911.124268
Epoch  350 | Loss=1398.780029
Epoch  400 | Loss=1004.966797
Epoch  450 | Loss=709.636108
Epoch  500 | Loss=492.109222
Epoch  550 | Loss=335.074524
Epoch  600 | Loss=224.226990
Epoch  650 | Loss=147.456558
Epoch  700 | Loss=97.942879
Epoch  750 | Loss=60.413593
Epoch  800 | Loss=37.137405
Epoch  850 | Loss=23.713711
Epoch  900 | Loss=15.750278
Epoch  950 | Loss=11.010330
Epoch 1000 | Loss=14.469288
Epoch 1050 | Loss=6.972051
Epoch 1100 | Loss=6.124888
Epoch 1150 | Loss=5.676047
Epoch 1200 | Lo

KeyboardInterrupt: 

In [14]:
model.eval()
with torch.no_grad():
    tau_pred = model(nu_dot_test, nu_test, eta_test)

    # ======================================================
    # === 1. Identify samples where any DOF is commanded
    # ======================================================
    active_sample_mask = torch.any(torch.abs(y_test) > 1e-6, dim=1)

    # ======================================================
    # === 2. Identify DOFs that were actually excited
    # ======================================================
    max_force_per_axis, _ = torch.max(torch.abs(y_test[active_sample_mask]), dim=0)
    dof_mask = max_force_per_axis > 1e-6

    # ======================================================
    # === 3. Global RMSE (all samples, all DOFs)
    # ======================================================
    mse = F.mse_loss(tau_pred, y_test, reduction="mean")
    rmse = torch.sqrt(mse)

    print("=== Test Set Error Metrics ===")
    print(f"MSE (all samples):  {mse.item():.3f} N²")
    print(f"RMSE (all samples): {rmse.item():.3f} N  (mean abs error ≈ ±{rmse.item():.2f} N)")

    # --- Per-axis RMSE (still over all samples) ---
    per_axis_rmse = torch.sqrt(torch.mean((tau_pred - y_test) ** 2, dim=0))
    print(f"Per-axis RMSE [N]: {per_axis_rmse.cpu().numpy()}")

    # ======================================================
    # === 4. Relative RMSEs using NONZERO commanded force only
    # ======================================================
    eps = 1e-6
    y_active = y_test[active_sample_mask]

    # --- Mean force per axis using only nonzero command samples ---
    mean_force_nonzero = torch.full_like(max_force_per_axis, float("nan"))

    for i in range(y_active.shape[1]):
        mask_nonzero = torch.abs(y_active[:, i]) > 1e-6
        if torch.any(mask_nonzero):
            mean_force_nonzero[i] = torch.mean(torch.abs(y_active[mask_nonzero, i]))

    # --- Relative RMSE metrics ---
    rel_rmse_nonzero = torch.full_like(per_axis_rmse, float("nan"))
    rel_rmse_max     = torch.full_like(per_axis_rmse, float("nan"))

    rel_rmse_nonzero[dof_mask] = (
        100 * per_axis_rmse[dof_mask] / (mean_force_nonzero[dof_mask] + eps)
    )
    rel_rmse_max[dof_mask] = (
        100 * per_axis_rmse[dof_mask] / (max_force_per_axis[dof_mask] + eps)
    )

    print(f"Per-axis RMSE relative to nonzero mean force [%]: {rel_rmse_nonzero.cpu().numpy()}")
    print(f"Per-axis RMSE relative to max force [%]: {rel_rmse_max.cpu().numpy()}")

    # ======================================================
    # === 5. Active-axis summary
    # ======================================================
    active_rmse_mean = torch.mean(per_axis_rmse[dof_mask])
    print(f"\nActive-axis mean RMSE: {active_rmse_mean.item():.3f} N")


=== Test Set Error Metrics ===
MSE (all samples):  0.001 N²
RMSE (all samples): 0.034 N  (mean abs error ≈ ±0.03 N)
Per-axis RMSE [N]: [3.36307054e-03 4.80524078e-03 4.92925989e-04 1.04438994e-04
 9.96559083e-06 8.18767622e-02]
Per-axis RMSE relative to nonzero mean force [%]: [       nan        nan        nan        nan        nan 0.96682125]
Per-axis RMSE relative to max force [%]: [       nan        nan        nan        nan        nan 0.51523304]

Active-axis mean RMSE: 0.082 N


In [178]:
lin_names = {"X_u", "Y_v", "Z_w", "K_p", "M_q", "N_r"}

with torch.no_grad():
    for name, param in model.named_parameters():
        val = param.item()

        # If you're using lin_params = -exp(raw) in build_damping,
        # then the effective physical coefficient is:
        if name in lin_names:
            val_eff = (-torch.exp(param)).item()
            print(f"{name}: raw={val:+.6f}  effective={val_eff:+.6f}")
        else:
            print(f"{name}: {val:+.6f}")


X_dot_u: +17.020849
Y_dot_v: +18.162828
Z_dot_w: -42.100353
K_dot_p: -0.599334
M_dot_q: -1.511613
N_dot_r: -1.324723
I_xx: +0.406418
I_yy: +1.339687
I_zz: +1.461428
x_g: +0.002780
y_g: +0.000144
z_g: -0.043963
X_u: raw=-0.094144  effective=-0.910152
Y_v: raw=-0.015289  effective=-0.984828
Z_w: raw=+0.005483  effective=-1.005498
K_p: raw=-0.025679  effective=-0.974648
M_q: raw=+0.004822  effective=-1.004833
N_r: raw=-0.014581  effective=-0.985524
X_uu: -30.971899
Y_vv: -90.984146
Z_ww: -108.878899
K_pp: +2.605973
M_qq: -4.485112
N_rr: -3.512359
B: +233.703033
z_b: -0.049154


In [179]:
# ============================================================
# === MAP vs MLE parameter shift diagnostics =================
# ============================================================

# MLE reference values (ground truth for comparison)
mle_vals = {


    "B": 236.00,
    "I_xx": 0.5,
    "I_yy": 1.76,
    "I_zz": 2.13,

    # CG / CB (m)
    "x_cg": 0.0, "y_cg": 0.0, "z_cg": 0.0,
    "x_cb": 0.0, "y_cb": 0.0, "z_b": -0.03,

    # Added mass
    "X_dot_u": -33.61,
    "Y_dot_v": -31.56,
    "Z_dot_w": -79.58,
    "K_dot_p": -0.1,
    "M_dot_q": -0.46,
    "N_dot_r": -0.70,

    # Linear & quadratic damping
    "X_u": -16.49, "X_uu": -42.49,
    "Y_v": -34.05, "Y_vv": -108.74,
    "Z_w": -35.66, "Z_ww": -128.31,
    "K_p": -1.24, "K_pp": -0.08,
    "M_q": -2.08, "M_qq": -1.61,
    "N_r": -2.88, "N_rr": -2.69,
}


lin_names = {"X_u", "Y_v", "Z_w", "K_p", "M_q", "N_r"}
eps = 1e-6

print("\n=== MAP vs MLE parameter shift ===")
print(f"{'Param':>8s} | {'MLE':>10s} | {'MAP (eff)':>10s} | {'Δabs':>10s} | {'Δrel':>8s}")
print("-" * 60)

with torch.no_grad():
    for name, param in model.named_parameters():
        if name not in mle_vals:
            continue

        mle = mle_vals[name]

        # Effective MAP value
        if name in lin_names:
            map_eff = (-torch.exp(param)).item()
        else:
            map_eff = param.item()

        delta_abs = abs(map_eff - mle)
        delta_rel = delta_abs / (abs(mle) + eps)

        print(
            f"{name:>8s} | "
            f"{mle:>10.4f} | "
            f"{map_eff:>10.4f} | "
            f"{delta_abs:>10.4f} | "
            f"{delta_rel:>8.2%}"
        )

print("=================================\n")



=== MAP vs MLE parameter shift ===
   Param |        MLE |  MAP (eff) |       Δabs |     Δrel
------------------------------------------------------------
 X_dot_u |   -33.6100 |    17.0208 |    50.6308 |  150.64%
 Y_dot_v |   -31.5600 |    18.1628 |    49.7228 |  157.55%
 Z_dot_w |   -79.5800 |   -42.1004 |    37.4796 |   47.10%
 K_dot_p |    -0.1000 |    -0.5993 |     0.4993 |  499.33%
 M_dot_q |    -0.4600 |    -1.5116 |     1.0516 |  228.61%
 N_dot_r |    -0.7000 |    -1.3247 |     0.6247 |   89.25%
    I_xx |     0.5000 |     0.4064 |     0.0936 |   18.72%
    I_yy |     1.7600 |     1.3397 |     0.4203 |   23.88%
    I_zz |     2.1300 |     1.4614 |     0.6686 |   31.39%
     X_u |   -16.4900 |    -0.9102 |    15.5798 |   94.48%
     Y_v |   -34.0500 |    -0.9848 |    33.0652 |   97.11%
     Z_w |   -35.6600 |    -1.0055 |    34.6545 |   97.18%
     K_p |    -1.2400 |    -0.9746 |     0.2654 |   21.40%
     M_q |    -2.0800 |    -1.0048 |     1.0752 |   51.69%
     N_r |    -2.8